In [14]:
# Cell 1 — Import + khai báo file input
from pathlib import Path

INPUT_FILE = Path("input/3352.docx")
OUTPUT_FILE = Path("output/3352.docx")

print("Input:", INPUT_FILE.resolve())
print("Exists:", INPUT_FILE.exists())
print("Output:", OUTPUT_FILE.resolve())

Input: D:\uit\code\document-pipeline\input\3352.docx
Exists: True
Output: D:\uit\code\document-pipeline\output\3352.docx


In [15]:
# Cell 2 — Đọc DOCX và thống kê cấu trúc

from docx import Document

doc = Document(INPUT_FILE)

print(f"Paragraphs: {len(doc.paragraphs)}")
print(f"Tables: {len(doc.tables)}")

Paragraphs: 26
Tables: 2


In [16]:
# Cell 3 — Inspect toàn bộ tables
for i, table in enumerate(doc.tables):
    print(f"\n{'=' * 80}")
    print(f"TABLE {i} | rows={len(table.rows)} | cols={len(table.columns)}")
    print("=" * 80)

    for row in table.rows:
        values = [cell.text.strip().replace("\n", " | ") for cell in row.cells]
        print(" | ".join(values))


TABLE 0 | rows=6 | cols=4
Tên chức danh | Tên chức danh tiếng Anh | Phân luồng | Phân loại
Tổng Giám đốc | Chief Executive Officer | Kinh doanh | trực tiếp | Nhân sự | quản lý
Phó Tổng Giám đốc thường trực | Standing Deputy Chief Executive Officer | Kinh doanh | Nhân sự
 |  | trực tiếp/ | quản lý
 |  | Hỗ trợ kinh | 
 |  | doanh | 

TABLE 1 | rows=3 | cols=4
Stt | Tên bước | Nội dung                 Chi tiết | Nội dung                 Chi tiết
1 | Khởi tạo giải ngân | Người chịu trách     CVKH/Chức danh khác theo quy định trong từng thời kỳ. | nhiệm | Người chịu trách     CVKH/Chức danh khác theo quy định trong từng thời kỳ. | nhiệm
1 | Khởi tạo giải ngân | Mô tả | Kiểm tra hồ sơ của KH cung cấp, đảm bảo hồ sơ đầy đủ theo Danh mục hồ sơ và thông tin hồ sơ chính xác, hợp lệ được hướng dẫn tại PL06.3352.COD, PL07.3352.COD quy trình này. | Trường hợp hồ sơ soạn thảo do ĐVKD thực hiện trước khi triển khai mô hình VHTD tập trung: Rà soát bổ sung đầy đủ chứng từ theo PL06.3351.COD tại Quy t

In [17]:
# Cell 4 — Detect table cần LLM
tables_need_llm = []
tables_skipped = []

for i, table in enumerate(doc.tables):
    rows = [
        [cell.text.strip() for cell in row.cells]
        for row in table.rows
    ]

    # Ghép toàn bộ nội dung table để kiểm tra
    table_text = " ".join(
        cell
        for row in rows
        for cell in row
        if cell
    )

    row_count = len(rows)
    col_count = len(table.columns)

    # Tạm thời: table có từ 2 dòng trở lên và có nội dung
    # sẽ được đưa vào nhóm cần review bằng LLM.
    if row_count >= 2 and table_text:
        tables_need_llm.append({
            "table_index": i,
            "rows": row_count,
            "cols": col_count,
            "text": table_text
        })
    else:
        # Lưu table bị skip để log chi tiết
        if row_count < 2:
            reason = "Less than 2 rows"
        elif not table_text:
            reason = "Empty table"
        else:
            reason = "Does not meet LLM criteria"

        tables_skipped.append({
            "table_index": i,
            "rows": row_count,
            "cols": col_count,
            "reason": reason
        })


# =========================
# Summary
# =========================

print(f"Total tables       : {len(doc.tables)}")
print(f"Tables need LLM    : {len(tables_need_llm)}")
print(f"Tables skipped     : {len(tables_skipped)}")

# =========================
# Tables need LLM
# =========================

print("\n--- Tables need LLM ---")

for item in tables_need_llm:
    print(
        f"Table {item['table_index']}: "
        f"{item['rows']} rows x {item['cols']} cols"
    )

# =========================
# Tables skipped
# =========================

print("\n--- Tables skipped ---")

for item in tables_skipped:
    print(
        f"Table {item['table_index']}: "
        f"{item['rows']} rows x {item['cols']} cols"
    )
    print(f"  Reason: {item['reason']}")

Total tables       : 2
Tables need LLM    : 2
Tables skipped     : 0

--- Tables need LLM ---
Table 0: 6 rows x 4 cols
Table 1: 3 rows x 4 cols

--- Tables skipped ---


In [18]:
# Cell 5 — Chuẩn hóa dữ liệu table để chuẩn bị gửi LLM
import json

tables_for_llm = []

for item in tables_need_llm:
    table = doc.tables[item["table_index"]]

    rows = [
        [cell.text.strip() for cell in row.cells]
        for row in table.rows
    ]

    tables_for_llm.append({
        "table_index": item["table_index"],
        "rows": rows
    })

print(f"Tables prepared for LLM: {len(tables_for_llm)}")

print(
    json.dumps(
        tables_for_llm[0],
        ensure_ascii=False,
        indent=2
    )
)

Tables prepared for LLM: 2
{
  "table_index": 0,
  "rows": [
    [
      "Tên chức danh",
      "Tên chức danh tiếng Anh",
      "Phân luồng",
      "Phân loại"
    ],
    [
      "Tổng Giám đốc",
      "Chief Executive Officer",
      "Kinh doanh\ntrực tiếp",
      "Nhân sự\nquản lý"
    ],
    [
      "Phó Tổng Giám đốc thường trực",
      "Standing Deputy Chief Executive Officer",
      "Kinh doanh",
      "Nhân sự"
    ],
    [
      "",
      "",
      "trực tiếp/",
      "quản lý"
    ],
    [
      "",
      "",
      "Hỗ trợ kinh",
      ""
    ],
    [
      "",
      "",
      "doanh",
      ""
    ]
  ]
}


In [19]:
# Cell 6 — Tạo prompt tối giản cho 1 table
table_data = tables_for_llm[0]

table_index = table_data["table_index"]
rows = table_data["rows"]

table_json = json.dumps(
    rows,
    ensure_ascii=False,
    indent=2
)

prompt = f"""
Chuyển bảng dưới đây thành semantic text để chatbot có thể hiểu và tìm kiếm nội dung.

Yêu cầu:
- Giữ nguyên đầy đủ thông tin trong bảng.
- Không tự suy diễn hoặc thêm thông tin.
- Không bỏ sót dòng/cột có nội dung.
- Diễn đạt thành các câu văn tự nhiên, dễ search bằng semantic search.
- Không cần giải thích cách xử lý.
- Chỉ trả về phần semantic text.

TABLE:
{table_json}
"""

print(prompt)


Chuyển bảng dưới đây thành semantic text để chatbot có thể hiểu và tìm kiếm nội dung.

Yêu cầu:
- Giữ nguyên đầy đủ thông tin trong bảng.
- Không tự suy diễn hoặc thêm thông tin.
- Không bỏ sót dòng/cột có nội dung.
- Diễn đạt thành các câu văn tự nhiên, dễ search bằng semantic search.
- Không cần giải thích cách xử lý.
- Chỉ trả về phần semantic text.

TABLE:
[
  [
    "Tên chức danh",
    "Tên chức danh tiếng Anh",
    "Phân luồng",
    "Phân loại"
  ],
  [
    "Tổng Giám đốc",
    "Chief Executive Officer",
    "Kinh doanh\ntrực tiếp",
    "Nhân sự\nquản lý"
  ],
  [
    "Phó Tổng Giám đốc thường trực",
    "Standing Deputy Chief Executive Officer",
    "Kinh doanh",
    "Nhân sự"
  ],
  [
    "",
    "",
    "trực tiếp/",
    "quản lý"
  ],
  [
    "",
    "",
    "Hỗ trợ kinh",
    ""
  ],
  [
    "",
    "",
    "doanh",
    ""
  ]
]



In [20]:
# Cell 7 — Load local.settings.json + tạo OpenAI client
import json
from openai import OpenAI

with open("local.settings.json", "r", encoding="utf-8") as f:
    settings = json.load(f)["Values"]

AZURE_OPENAI_API_KEY = settings["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_DEPLOYMENT = settings["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_ENDPOINT = settings["AZURE_OPENAI_ENDPOINT"]

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT
)

print("OpenAI client initialized")
print("Model:", AZURE_OPENAI_DEPLOYMENT)


OpenAI client initialized
Model: gpt-5.4-mini-route


In [21]:
results = []

for table_data in tables_for_llm:
    table_index = table_data["table_index"]
    rows = table_data["rows"]

    table_json = json.dumps(
        rows,
        ensure_ascii=False,
        indent=2
    )


    prompt = f"""
Bạn là hệ thống chuyển đổi bảng dữ liệu trong tài liệu nghiệp vụ
thành văn bản có cấu trúc để sử dụng làm dữ liệu cho chatbot/RAG.

NHIỆM VỤ

Đọc bảng đầu vào và chuyển toàn bộ thông tin trong bảng thành
structured semantic text.

Mục tiêu là:
- Giữ nguyên thông tin của bảng.
- Làm rõ quan hệ giữa các trường và giá trị.
- Loại bỏ sự phụ thuộc vào cấu trúc bảng khi chatbot tìm kiếm.
- Cho phép người đọc hiểu được nội dung mà không cần nhìn bảng gốc.

QUY TRÌNH XỬ LÝ BẮT BUỘC

Bước 1 — Xác định record

Hãy coi MỖI DÒNG của bảng là một record độc lập.

Không được gộp các giá trị của các dòng khác nhau ở bước này.

Bước 2 — Xác định entity của từng record

Từ các cột trong bảng, xác định các trường tạo nên "đối tượng chính"
của record.

Các trường mô tả định danh hoặc nhận diện đối tượng phải được giữ
nguyên cùng nhau.

Ví dụ:
- tên đối tượng
- mã đối tượng
- tên tiếng Anh
- số hiệu
- hoặc các trường khác có vai trò nhận diện đối tượng.

Không được tự động coi một cột đơn lẻ là entity nếu điều đó làm
mất quan hệ giữa các cột.

Bước 3 — Group record

Chỉ group hai hoặc nhiều record khi chúng mô tả CÙNG MỘT ĐỐI TƯỢNG.

Hai record KHÔNG được coi là cùng đối tượng nếu các giá trị định danh
chính của chúng khác nhau.

Đặc biệt:

KHÔNG được group toàn bộ bảng theo từng cột.

KHÔNG được lấy tất cả giá trị của cột A gom lại với tất cả giá trị
của cột B.

KHÔNG được tạo kết quả kiểu:

Thuộc tính A: A1 / A2
Thuộc tính B: B1 / B2
Thuộc tính C: C1 / C2 / C3

nếu các giá trị này thuộc các record khác nhau.

Bước 4 — Merge thuộc tính

Sau khi đã xác định các record thuộc CÙNG MỘT ĐỐI TƯỢNG,
mới được merge các giá trị khác nhau của cùng một thuộc tính.

Ví dụ:

Record 1:
Entity A | X | Value 1

Record 2:
Entity A | X | Value 2

=> merge thành:

Entity: A
X: Value 1 / Value 2

Nhưng:

Record 1:
Entity A | X | Value 1

Record 2:
Entity B | X | Value 2

=> KHÔNG được merge.

Phải tạo hai block riêng.

Bước 5 — Xuất kết quả

Mỗi entity chỉ có MỘT block.

Trong một block:
- Mỗi thuộc tính chỉ xuất hiện một lần.
- Nếu thuộc tính có nhiều giá trị do nhiều record của cùng entity,
  merge các giá trị bằng "/ ".
- Các thuộc tính khác nhau giữa các entity tuyệt đối không được
  trộn lẫn với nhau.

QUY TẮC QUAN TRỌNG NHẤT

Thứ tự xử lý bắt buộc là:

ROWS
→ IDENTIFY ENTITY
→ GROUP ROWS BY ENTITY
→ MERGE ATTRIBUTES WITHIN EACH ENTITY
→ OUTPUT BLOCKS

Tuyệt đối KHÔNG xử lý theo:

COLUMNS
→ MERGE VALUES
→ OUTPUT

Nếu không chắc hai record có cùng entity hay không,
KHÔNG được merge chúng.

YÊU CẦU VỀ KHẢ NĂNG SEARCH

Kết quả phải giúp chatbot có thể tìm kiếm riêng biệt các thông tin
quan trọng trong bảng.

Do đó:
- Các đối tượng khác nhau phải có thể phân biệt được.
- Các thuộc tính quan trọng phải được thể hiện rõ.
- Các giá trị thuộc cùng một đối tượng phải được liên kết rõ ràng.
- Không làm mất thông tin chỉ vì thông tin đó bị lặp lại giữa
  nhiều dòng.
- Khi nhiều giá trị thuộc cùng một thuộc tính của cùng một đối tượng,
  phải đặt các giá trị đó trên cùng một dòng thuộc tính.

KIỂM TRA TRƯỚC KHI TRẢ KẾT QUẢ

Trước khi trả về kết quả, phải kiểm tra:

- Có bỏ sót thông tin nào trong bảng không?
- Có tạo thêm thông tin không có trong bảng không?
- Các giá trị trên cùng một dòng có còn đúng quan hệ không?
- Các dòng thuộc cùng một đối tượng đã được gộp đúng chưa?
- Trong mỗi block có thuộc tính nào bị lặp lại không?
- Nếu một thuộc tính có nhiều giá trị, các giá trị đó đã được
  merge vào cùng một dòng chưa?

Nếu phát hiện một thuộc tính xuất hiện nhiều lần trong cùng một block,
phải gộp các giá trị của thuộc tính đó thành một dòng duy nhất
trước khi trả kết quả.

BẢNG ĐẦU VÀO:


{table_json}
"""

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": (
                    "Bạn là chuyên gia chuyển đổi dữ liệu bảng "
                    "nghiệp vụ thành structured semantic text "
                    "cho hệ thống RAG."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    semantic_text = response.choices[0].message.content.strip()

    results.append({
        "table_index": table_index,
        "semantic_text": semantic_text
    })

    print(f"\n{'=' * 80}")
    print(f"TABLE {table_index}")
    print("=" * 80)
    print(semantic_text)


TABLE 0
**Block 1 — Tên chức danh: Tổng Giám đốc**

- **Tên chức danh tiếng Anh:** Chief Executive Officer
- **Phân luồng:** Kinh doanh trực tiếp
- **Phân loại:** Nhân sự quản lý

**Block 2 — Tên chức danh: Phó Tổng Giám đốc thường trực**

- **Tên chức danh tiếng Anh:** Standing Deputy Chief Executive Officer
- **Phân luồng:** Kinh doanh trực tiếp / Hỗ trợ kinh doanh
- **Phân loại:** Nhân sự quản lý

TABLE 1
**Block 1 — Khởi tạo giải ngân**

- **Stt:** 1
- **Tên bước:** Khởi tạo giải ngân
- **Người chịu trách nhiệm:** CVKH / Chức danh khác theo quy định trong từng thời kỳ
- **Mô tả:** Kiểm tra hồ sơ của KH cung cấp, đảm bảo hồ sơ đầy đủ theo Danh mục hồ sơ và thông tin hồ sơ chính xác, hợp lệ được hướng dẫn tại PL06.3352.COD, PL07.3352.COD quy trình này. Trường hợp hồ sơ soạn thảo do ĐVKD thực hiện trước khi triển khai mô hình VHTD tập trung: rà soát bổ sung đầy đủ chứng từ theo PL06.3351.COD tại Quy trình soạn thảo hồ sơ tín dụng. Nếu chưa đầy đủ, chính xác, hợp lệ: làm việc với KH đ

In [22]:
# CELL 9 — Validate LLM results

print(f"Expected tables : {len(tables_for_llm)}")
print(f"LLM results     : {len(results)}")

assert len(results) == len(tables_for_llm), (
    f"Missing results: expected {len(tables_for_llm)}, got {len(results)}"
)

for result in results:
    table_index = result["table_index"]
    semantic_text = result["semantic_text"]

    assert semantic_text, (
        f"Table {table_index} has empty semantic text"
    )

    print(f"\n{'=' * 80}")
    print(f"TABLE {table_index} — VALIDATED")
    print("=" * 80)
    print(semantic_text)

print("\n✓ All table results are valid.")

Expected tables : 2
LLM results     : 2

TABLE 0 — VALIDATED
**Block 1 — Tên chức danh: Tổng Giám đốc**

- **Tên chức danh tiếng Anh:** Chief Executive Officer
- **Phân luồng:** Kinh doanh trực tiếp
- **Phân loại:** Nhân sự quản lý

**Block 2 — Tên chức danh: Phó Tổng Giám đốc thường trực**

- **Tên chức danh tiếng Anh:** Standing Deputy Chief Executive Officer
- **Phân luồng:** Kinh doanh trực tiếp / Hỗ trợ kinh doanh
- **Phân loại:** Nhân sự quản lý

TABLE 1 — VALIDATED
**Block 1 — Khởi tạo giải ngân**

- **Stt:** 1
- **Tên bước:** Khởi tạo giải ngân
- **Người chịu trách nhiệm:** CVKH / Chức danh khác theo quy định trong từng thời kỳ
- **Mô tả:** Kiểm tra hồ sơ của KH cung cấp, đảm bảo hồ sơ đầy đủ theo Danh mục hồ sơ và thông tin hồ sơ chính xác, hợp lệ được hướng dẫn tại PL06.3352.COD, PL07.3352.COD quy trình này. Trường hợp hồ sơ soạn thảo do ĐVKD thực hiện trước khi triển khai mô hình VHTD tập trung: rà soát bổ sung đầy đủ chứng từ theo PL06.3351.COD tại Quy trình soạn thảo hồ sơ

In [23]:
# CELL 10 — Replace tables with semantic text

from docx.oxml import OxmlElement
from docx.text.paragraph import Paragraph


def insert_paragraph_before_table(table, text):
    """
    Insert semantic text as paragraphs immediately before the table.
    Then remove the original table.
    """
    table_element = table._element

    # Insert one paragraph for each line of semantic text
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    for line in lines:
        p = OxmlElement("w:p")
        table_element.addprevious(p)

        paragraph = Paragraph(p, table._parent)

        run = paragraph.add_run(line)

    # Remove original table
    table_element.getparent().remove(table_element)


# Map table_index -> semantic_text
semantic_text_by_table = {
    result["table_index"]: result["semantic_text"]
    for result in results
}

# Process tables in reverse order so original table indexes remain stable
for table_index in sorted(semantic_text_by_table.keys(), reverse=True):
    table = doc.tables[table_index]
    semantic_text = semantic_text_by_table[table_index]

    insert_paragraph_before_table(
        table,
        semantic_text
    )

print("✓ All selected tables were replaced with semantic text.")
print(f"Tables processed: {len(semantic_text_by_table)}")
print(f"Remaining tables: {len(doc.tables)}")

✓ All selected tables were replaced with semantic text.
Tables processed: 2
Remaining tables: 0


In [ ]:
# Cell 11 — Export processed DOCX

# Tạo thư mục output nếu chưa tồn tại
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Lưu document đã được xử lý
doc.save(OUTPUT_FILE)

print(f"✓ Processed DOCX saved to: {OUTPUT_FILE}")

✓ Processed DOCX saved to: output\3352.docx
